<a href="https://colab.research.google.com/github/PAJILAPIP/BigData26_B_2411531008_MuhammadFazilAfif/blob/main/Praktikum2/BD_B_P02_2411531008_MuhammadFazilAfif.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.9 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

In [3]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))



Mounted at /content/drive
['transaksi_bersih.csv']


In [4]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar}  ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()

    if random.random() < 0.2:
        kategori = kategori.upper() + "   "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)

df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


In [5]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


In [6]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris sebelum drop_duplicates():", len(df))

Jumlah baris sebelum drop_duplicates(): 495


In [7]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


In [8]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [9]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [10]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [11]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

In [12]:
file_path = os.path.join(DIR_SIMPAN, "transaksi_bersih.csv")
df.to_csv(file_path, index=False)
print("Dataset bersih tersimpan:", len(df), "baris di", file_path)

Dataset bersih tersimpan: 490 baris di /content/drive/MyDrive/BigData/Praktikum2/transaksi_bersih.csv


# LATIHAN

In [15]:
df["is_valid_price"] = df["price"] > 0


print(df["is_valid_price"].value_counts())


invalid_prices = df[~df["is_valid_price"]]
if not invalid_prices.empty:
    print("\nData dengan harga tidak valid:")
    print(invalid_prices[["transaction_id", "price"]])
else:
    print("\nTidak ada data dengan harga tidak valid (<= 0).")

is_valid_price
True    490
Name: count, dtype: int64

Tidak ada data dengan harga tidak valid (<= 0).


In [14]:
kategori_counts = df["category"].value_counts()

print("Jumlah transaksi per Kategori Produk:")
print(kategori_counts)

Jumlah transaksi per Kategori Produk:
category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64


# Studi Kasus

 1. Mengapa Kedua Angka Bisa Berbeda: Angka 515 milik tim IT adalah jumlah baris pada transaksi_mentah.csv, yaitu data mentah hasil acquisition langsung dari “sistem” sebelum melalui proses apa pun, sehingga masih memuat 15 baris transaksi yang tercatat dua kali (duplicate, akibat simulasi kegagalan jaringan pada K-2) dan 10 baris yang kehilangan data wajib berupa customer_name atau payment_method. Angka 490 milik tim Finance adalah jumlah baris pada transaksi_bersih.csv, yaitu hasil setelah pipeline pra-pemrosesan pada K-3 dan K-4 membuang 25 baris tersebut (5 duplicate + 20 data wajib kosong). Kedua angka sama-sama benar untuk tujuan masing-masing sistem yang mencatatnya, namun mewakili tahap pipeline yang berbeda: 515 adalah representasi Volume mentah, sedangkan 490 adalah representasi setelah Veracity-nya diverifikasi.

2. Apakah 490 “Lebih Benar” daripada 515?: Ya, untuk keperluan pelaporan finansial (menghitung total penjualan atau performa bulanan), 490 baris lebih dapat dipercaya (lebih tinggi Veracity-nya) dibanding 515 baris, karena 515 masih memuat 5 transaksi yang sebenarnya adalah kejadian yang sama namun tercatat dua kali, jika ikut dihitung, penjualan bulan tersebut akan digelembungkan (overcounted) sebesar transaksi duplikat itu. Namun “lebih benar” di sini bukan berarti 515 salah atau tidak berguna: angka 515 tetap penting bagi tim IT untuk mendiagnosis masalah infrastruktur, misalnya untuk menyelidiki mengapa kegagalan jaringan menyebabkan 5 pengiriman ulang data. Intinya, Veracity bukan sekadar “angka mana yang benar”, melainkan “angka mana yang valid untuk tujuan analisis tertentu”, 490 valid untuk pelaporan penjualan, sedangkan 515 valid untuk audit teknis pipeline data.

3. Menjelaskan Keputusan Membiarkan Kolom Rating Tetap Kosong: Kepada tim Finance yang ingin tahu “rating rata-rata semua transaksi”, penjelasannya adalah: dari 490 transaksi bersih, sebagian besar (166 dari 515 baris mentah, atau sekitar sepertiga transaksi) memang tidak memiliki rating sama sekali karena rating bersifat opsional, pembeli tidak diwajibkan memberi rating. Mengisi kekosongan tersebut dengan angka tebakan (misalnya nilai tengah 3, atau rata-rata dari transaksi yang memiliki rating) akan membuat “rating rata-rata” menjadi angka buatan yang seolah-olah mencerminkan opini pelanggan, padahal sebagian besar pelanggan tersebut tidak pernah benar-benar menyatakan opininya. Rekomendasi yang lebih jujur adalah tetap melaporkan dua angka sekaligus: rata-rata rating hanya dari transaksi yang benar-benar memiliki rating, dan persentase transaksi yang tidak memiliki rating sama sekali, sehingga tim Finance memahami konteks di balik angka tersebut, alih-alih menerima satu angka rata-rata yang menyembunyikan ketidaklengkapan data.